#  BeautifulSoup & Selenium을 활용한 데이터 수집 및 저장

##  웹 스크래핑 개요 및 학습 목표

###  웹 스크래핑(Web Scraping)이란?
웹 스크래핑은 **웹 사이트에서 데이터를 자동으로 추출하는 기술**입니다. 
이 과정에서 `BeautifulSoup`과 `Selenium` 같은 라이브러리를 사용하여 원하는 정보를 가져올 수 있습니다.

###  학습 목표
- `BeautifulSoup`을 활용한 정적 웹 페이지 데이터 추출 방법 이해
- `Selenium`을 이용한 동적 웹 페이지 데이터 수집 방법 익히기
- 수집한 데이터를 `pandas`를 활용하여 저장하고 활용하는 방법 익히기

##  웹 스크래핑 환경 설정

###  필수 라이브러리 설치

In [ ]:
pip install requests beautifulsoup4 selenium

###  라이브러리 불러오기

In [ ]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
import pandas as pd

##  `BeautifulSoup`을 활용한 정적 웹 페이지 크롤링

###  웹 페이지 요청 및 HTML 파싱

In [ ]:
url = "https://naver.com"
response = requests.get(url)    # requests 라이브러리를 사용해 웹 페이지 요청을 보냄(HTTP 응답 객체를 받는다.)
soup = BeautifulSoup(response.text, "html.parser")  # 서버에서 받은 HTML 코드(문자열)를 가져옴, HTML을 분석하고 다룰 수 있도록 변환

###  특정 데이터 추출 (`find()`, `find_all()` 활용)

In [ ]:
# 특정 태그 가져오기
h1_tag = soup.find("title").text
print("제목:", h1_tag)

# 여러 개의 태그 가져오기
all_links = soup.find_all("a")
for link in all_links:
    print(link.text, link["href"])

##  `Selenium`을 활용한 동적 웹 페이지 크롤링

###  웹 드라이버 설정 및 실행

In [ ]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

# 크롬 드라이버 자동 설치 및 실행
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# 웹 페이지 열기
driver.get("https://www.google.com")
time.sleep(3)  # 페이지 로드 대기


###  특정 요소 찾기 및 상호작용

In [ ]:
search_box = driver.find_element(By.NAME, "q")  # 요소의 name 속성 값을 기준으로 검색 -> 구글 검색창의 <input name="q"> 이다.
search_box.send_keys("Python 웹 스크래핑")  # 검색어 입력
search_box.send_keys(Keys.RETURN)  # 엔터 키 입력

##  수집한 데이터 저장 및 활용

In [ ]:
data = {"제목": ["Python 기초", "웹 크롤링", "데이터 분석"],
        "링크": ["https://example.com/1", "https://example.com/2", "https://example.com/3"]}

df = pd.DataFrame(data)

# CSV 파일로 저장
df.to_csv("scraped_data.csv", index=False, encoding="utf-8-sig")

- 네이버 뉴스 제목과 링크 수집

In [ ]:
import requests
from bs4 import BeautifulSoup

# 네이버 뉴스 웹페이지
url = "https://news.naver.com/"

# 웹 페이지 요청 및 HTML 파싱
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# 뉴스 제목과 링크 수집
news_list = []
for item in soup.select("a[href^='https://n.news.naver.com/article/']"):  # 뉴스 기사 링크 찾기
    title = item.get_text(strip=True)   
    link = item.get("href") # "href" 속성 값을 가져옴
    if title:  # 빈 제목 제외
        news_list.append((title, link))

# 출력
for title, link in news_list[:10]:  # 상위 10개 뉴스만 출력
    print(f"제목: {title}\n링크: {link}\n")

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# 크롬 드라이버 자동 설치 및 실행
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)


# 네이버 검색 페이지 열기
driver.get("https://www.naver.com")

# 검색창이 나타날 때까지 대기
search_box = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.NAME, "query"))
)

# 검색어 입력 후 실행
search_box.send_keys("Python 웹 스크래핑")
search_box.send_keys(Keys.RETURN)

# 검색 결과 로딩 대기
time.sleep(3)

# 검색 결과 제목 가져오기
titles = driver.find_elements(By.CSS_SELECTOR, "a.news_tit")

# 결과 출력
search_results = []
for title in titles[:5]:  # 상위 5개 결과만 출력
    search_results.append((title.text, title.get_attribute("href")))
    print(f"제목: {title.text}\n링크: {title.get_attribute('href')}\n")

# 브라우저 종료
driver.quit()

- 수집한 데이터 pandas 이용하여 csv 파일로 저장하기

In [ ]:
import pandas as pd

# 뉴스 데이터 예제 (위에서 수집한 데이터 사용)
df = pd.DataFrame(news_list, columns=["제목", "링크"])

# CSV 파일로 저장
df.to_csv("news_scraping_results.csv", index=False, encoding="utf-8-sig")

print("✅ 뉴스 데이터가 news_scraping_results.csv 파일로 저장되었습니다!")

In [ ]:
df_search = pd.DataFrame(search_results, columns=["제목", "링크"])

# CSV 파일로 저장
df_search.to_csv("naver_search_results.csv", index=False, encoding="utf-8-sig")

print("✅ 네이버 검색 결과가 naver_search_results.csv 파일로 저장되었습니다!")